In [0]:
#Load dataset
from pyspark.sql.functions import col

df = spark.table("workspace.default.sephora_reviews_enrichedsn")

display(df.limit(5))

In [0]:
#Basic text cleaning function. Standardize text to improve matching.
from pyspark.sql.functions import lower, regexp_replace, trim

def clean_text(c):
    return trim(
        regexp_replace(
            lower(col(c)),
            r"[^a-zA-Z0-9 ]",
            " "
        )
    )

In [0]:
#clean important text columns
df_clean = (
    df
    .withColumn("product_name_clean", clean_text("product_name"))
    .withColumn("brand_name_clean", clean_text("brand_name"))
    .withColumn("review_title_clean", clean_text("review_title"))
    .withColumn("review_text_clean", clean_text("review_text"))
    .withColumn("ingredients_clean", clean_text("ingredients"))
)

In [0]:
#aggregate reviews to product level. We create one profile per product.
from pyspark.sql.functions import concat_ws, collect_list, first

product_text_df = (
    df_clean
    .groupBy("product_id")
    .agg(
        first("product_name_clean").alias("product_name"),
        first("brand_name_clean").alias("brand_name"),
        first("ingredients_clean").alias("ingredients"),
        
        concat_ws(
            " ",
            collect_list("review_title_clean")
        ).alias("all_review_titles"),
        
        concat_ws(
            " ",
            collect_list("review_text_clean")
        ).alias("all_review_text")
    )
)

In [0]:
# create weighted combined text column.
# Included text fields:
# - product_name
# - brand_name
# - ingredients
# - all_review_titles
# - all_review_text
# Product / brand / ingredients are repeated lightly to strengthen product identity.

from pyspark.sql.functions import concat_ws

product_text_df = (
    product_text_df
    .withColumn(
        "combined_text",
        concat_ws(
            " ",
            # slightly upweight product identity / ingredients
            col("product_name"),
            col("product_name"),
            col("brand_name"),
            col("brand_name"),
            col("ingredients"),
            col("ingredients"),

            # supporting text signals
            col("all_review_titles"),
            col("all_review_text")
        )
    )
)

In [0]:
# Final dataset for recommender.
# Keep the source text fields as well so it is easy to verify what went into combined_text.
final_content_df = product_text_df.select(
    "product_id",
    "product_name",
    "brand_name",
    "ingredients",
    "all_review_titles",
    "all_review_text",
    "combined_text"
)

display(final_content_df.limit(5))

In [0]:
#Check dataset size
print("Number of products:", final_content_df.count())

In [0]:
#SAVE PREPARED DATASET
OUTPUT_TABLE = "workspace.default.sephora_content_prep"

spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")

final_content_df.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

print("Saved to:", OUTPUT_TABLE)

In [0]:
# TF-IDF preprocessing code (faster version)
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.sql.functions import col, size, array_distinct

# =========================================================
# 1) Load prepared content table
# =========================================================
content_df = spark.table("workspace.default.sephora_content_prep")

display(content_df.limit(5))


In [0]:
# =========================================================
# 2) Keep only rows with non-null combined_text
# =========================================================
content_df = content_df.filter(
    col("combined_text").isNotNull() & (col("combined_text") != "")
)

print("Rows after filtering:", content_df.count())

In [0]:
# =========================================================
# 3) Tokenize text
#    RegexTokenizer handles repeated spaces more cleanly.
# =========================================================
tokenizer = RegexTokenizer(
    inputCol="combined_text",
    outputCol="tokens",
    pattern=r"\W+",
    gaps=True,
    toLowercase=True
)

tokenized_df = tokenizer.transform(content_df)

display(tokenized_df.select("product_id", "combined_text", "tokens").limit(5))

In [0]:
# =========================================================
# 4) Remove stopwords
#    Add a few domain-generic review words that often add noise.
# =========================================================
default_stopwords = StopWordsRemover.loadDefaultStopWords("english")
custom_stopwords = [
    "sephora", "product", "products", "skin", "skincare", "makeup",
    "use", "used", "using", "like", "really", "nice", "good", "great"
]

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens",
    stopWords=default_stopwords + custom_stopwords
)

filtered_df = (
    remover.transform(tokenized_df)
    .withColumn("filtered_tokens", array_distinct(col("filtered_tokens")))
    .filter(size(col("filtered_tokens")) > 0)
)

display(filtered_df.select("product_id", "filtered_tokens").limit(5))

In [0]:
# =========================================================
# 5) Keep unigram tokens only for speed
#    This removes the biggest runtime cost introduced in the
#    previous revision (bigrams), while preserving the core
#    TF-IDF pipeline and all key text fields in combined_text.
# =========================================================
text_features_df = filtered_df.withColumn(
    "text_terms",
    col("filtered_tokens")
)

display(text_features_df.select("product_id", "text_terms").limit(5))


In [0]:
# =========================================================
# 6) Convert tokens into term-frequency vectors
#    Faster settings:
#    - unigram only
#    - smaller vocabulary than the previous revision
# =========================================================
cv = CountVectorizer(
    inputCol="text_terms",
    outputCol="raw_features",
    vocabSize=10000,
    minDF=2
)

cv_model = cv.fit(text_features_df)
featurized_df = cv_model.transform(text_features_df)

display(featurized_df.select("product_id", "raw_features").limit(5))


In [0]:
# =========================================================
# 7) Apply IDF and save final TF-IDF dataset
# =========================================================
idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)

idf_model = idf.fit(featurized_df)
tfidf_df = idf_model.transform(featurized_df)

tfidf_final_df = tfidf_df.select(
    "product_id",
    "product_name",
    "brand_name",
    "ingredients",
    "all_review_titles",
    "all_review_text",
    "combined_text",
    "filtered_tokens",
    "text_terms",
    "tfidf_features"
)

OUTPUT_TABLE = "workspace.default.sephora_tfidf_features"

spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")
tfidf_final_df.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

print("Saved TF-IDF table to:", OUTPUT_TABLE)
display(tfidf_final_df.select("product_id", "filtered_tokens", "tfidf_features").limit(5))


In [0]:
vocab = cv_model.vocabulary[:200]
print(vocab)

In [0]:
# Start Content Filter modelling. Load data
from pyspark.sql.functions import col

tfidf_df = spark.table("workspace.default.sephora_tfidf_features").select(
    "product_id",
    "product_name",
    "brand_name",
    "ingredients",
    "all_review_titles",
    "all_review_text",
    "combined_text",
    "tfidf_features"
)

display(tfidf_df.limit(5))
print("Number of products:", tfidf_df.count())

In [0]:
# move product vectors into Python memory
import numpy as np
import pandas as pd

# Collect to driver
pdf = tfidf_df.toPandas()

print("Shape of pandas table:", pdf.shape)
pdf.head()

In [0]:
#convert Spark vectors into numpy matrix
# Convert Spark ML vectors to dense numpy arrays
X = np.vstack(pdf["tfidf_features"].apply(lambda v: v.toArray()).values)

print("TF-IDF matrix shape:", X.shape)

In [0]:
# compute cosine similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(X, X)

print("Cosine similarity matrix shape:", cosine_sim.shape)

In [0]:
#create lookup helpers
# Reset index so matrix row index matches dataframe row index cleanly
pdf = pdf.reset_index(drop=True)

# Mapping from product_id to row index
product_id_to_idx = pd.Series(pdf.index, index=pdf["product_id"]).drop_duplicates()

# For product name lookup, lower-case for easier matching
pdf["product_name_lower"] = pdf["product_name"].fillna("").str.lower()
pdf["brand_name_lower"] = pdf["brand_name"].fillna("").str.lower()

In [0]:
#recommender by product_id
def recommend_by_product_id(product_id, top_n=10):
    if product_id not in product_id_to_idx:
        print(f"product_id '{product_id}' not found.")
        return None
    
    idx = product_id_to_idx[product_id]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the item itself
    sim_scores = [x for x in sim_scores if x[0] != idx][:top_n]
    
    rec_indices = [x[0] for x in sim_scores]
    rec_scores = [x[1] for x in sim_scores]
    
    recs = pdf.loc[rec_indices, ["product_id", "product_name", "brand_name"]].copy()
    recs["similarity_score"] = rec_scores
    
    source_row = pdf.loc[idx, ["product_id", "product_name", "brand_name"]]
    print("SOURCE PRODUCT")
    print(source_row.to_string())
    print("\nRECOMMENDATIONS")
    
    return recs.reset_index(drop=True)

In [0]:
#Example
recommend_by_product_id("P123456", top_n=10)

In [0]:
#recommender by product name
def recommend_by_product_name(product_name_query, top_n=10):
    q = product_name_query.lower().strip()
    
    matches = pdf[pdf["product_name_lower"].str.contains(q, na=False)].copy()
    
    if matches.empty:
        print(f"No product name match found for: {product_name_query}")
        return None
    
    # If multiple matches, take the first one
    matched_idx = matches.index[0]
    matched_row = pdf.loc[matched_idx, ["product_id", "product_name", "brand_name"]]
    
    sim_scores = list(enumerate(cosine_sim[matched_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the item itself
    sim_scores = [x for x in sim_scores if x[0] != matched_idx][:top_n]
    
    rec_indices = [x[0] for x in sim_scores]
    rec_scores = [x[1] for x in sim_scores]
    
    recs = pdf.loc[rec_indices, ["product_id", "product_name", "brand_name"]].copy()
    recs["similarity_score"] = rec_scores
    
    print("MATCHED SOURCE PRODUCT")
    print(matched_row.to_string())
    print("\nRECOMMENDATIONS")
    
    return recs.reset_index(drop=True)

In [0]:
# Example
recommend_by_product_name("vitamin c", top_n=10)

In [0]:
# better matcher when many similar names exist. Inspect all candidates first.
def search_products(product_name_query, top_n=20):
    q = product_name_query.lower().strip()
    
    matches = pdf[pdf["product_name_lower"].str.contains(q, na=False)][
        ["product_id", "product_name", "brand_name"]
    ].drop_duplicates().head(top_n)
    
    return matches.reset_index(drop=True)

In [0]:
#Example
search_products("vitamin c", top_n=20)

In [0]:
# save full similarity pairs for one selected product. Useful for report/demo.
def save_recommendations_for_product(product_name_query, top_n=10, output_table="workspace.default.sephora_content_recs_demo"):
    recs = recommend_by_product_name(product_name_query, top_n=top_n)
    
    if recs is None:
        return
    
    recs_spark = spark.createDataFrame(recs)
    
    spark.sql(f"DROP TABLE IF EXISTS {output_table}")
    recs_spark.write.mode("overwrite").saveAsTable(output_table)
    
    print(f"Saved recommendations to: {output_table}")
    display(recs_spark)

In [0]:
#Example
save_recommendations_for_product("vitamin c", top_n=10)

In [0]:
#show top recommendations nicely (up tp Cell 10 only)
recs = recommend_by_product_name("vitamin c", top_n=10)
recs

In [0]:
#load interaction data to form TRAIN TEST SPLIT
from pyspark.sql.functions import col

interactions_df = spark.table("workspace.default.sephora_reviews_enrichedsn").select(
    "author_id",
    "product_id",
    "rating"
)

display(interactions_df.limit(5))

print("Total interactions:", interactions_df.count())

In [0]:
# remove nulls
interactions_df = interactions_df.filter(
    col("author_id").isNotNull() &
    col("product_id").isNotNull()
)

print("Interactions after cleaning:", interactions_df.count())

In [0]:
# keep users with sufficient interactions. Users with too few interactions cannot be split meaningfully. We keep users with at least 5 interactions (recommended).
from pyspark.sql.functions import count

user_counts = (
    interactions_df
    .groupBy("author_id")
    .agg(count("*").alias("num_interactions"))
)

eligible_users = user_counts.filter(
    col("num_interactions") >= 5
)

filtered_interactions = interactions_df.join(
    eligible_users.select("author_id"),
    on="author_id",
    how="inner"
)

print("Remaining interactions:", filtered_interactions.count())
print("Remaining users:", eligible_users.count())

In [0]:
# per-user random ordering. We randomize interactions within each user.
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rand

window_spec = Window.partitionBy("author_id").orderBy(rand())

ranked_df = filtered_interactions.withColumn(
    "row_num",
    row_number().over(window_spec)
)

In [0]:
# compute split threshold per user
from pyspark.sql.functions import ceil

user_sizes = (
    ranked_df
    .groupBy("author_id")
    .agg(count("*").alias("num_interactions"))
)

ranked_df = ranked_df.join(user_sizes, on="author_id")

ranked_df = ranked_df.withColumn(
    "train_cutoff",
    ceil(col("num_interactions") * 0.8)
)

In [0]:
# create train and test datasets
train_df = ranked_df.filter(
    col("row_num") <= col("train_cutoff")
).drop("row_num", "num_interactions", "train_cutoff")

test_df = ranked_df.filter(
    col("row_num") > col("train_cutoff")
).drop("row_num", "num_interactions", "train_cutoff")

In [0]:
# confirm split proportions
print("Train interactions:", train_df.count())
print("Test interactions:", test_df.count())

print(
    "Train ratio:",
    train_df.count() / (train_df.count() + test_df.count())
)

In [0]:
#confirm each user appears in both sets
train_users = train_df.select("author_id").distinct().count()
test_users = test_df.select("author_id").distinct().count()

print("Users in train:", train_users)
print("Users in test:", test_users)

In [0]:
#save train and test tables
TRAIN_TABLE = "workspace.default.sephora_interactions_train"
TEST_TABLE = "workspace.default.sephora_interactions_test"

spark.sql(f"DROP TABLE IF EXISTS {TRAIN_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")

train_df.write.mode("overwrite").saveAsTable(TRAIN_TABLE)
test_df.write.mode("overwrite").saveAsTable(TEST_TABLE)

print("Saved train table:", TRAIN_TABLE)
print("Saved test table:", TEST_TABLE)

In [0]:
# START EVALUATION STAGE
# Load required tables
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TF-IDF features
tfidf_df = spark.table("workspace.default.sephora_tfidf_features").select(
    "product_id",
    "product_name",
    "tfidf_features"
)

# train/test interactions
train_df = spark.table("workspace.default.sephora_interactions_train")
test_df = spark.table("workspace.default.sephora_interactions_test")

In [0]:
#convert TF-IDF vectors to matrix
pdf_items = tfidf_df.toPandas().reset_index(drop=True)

X = np.vstack(
    pdf_items["tfidf_features"].apply(lambda v: v.toArray()).values
)

cosine_sim = cosine_similarity(X, X)

print("Similarity matrix shape:", cosine_sim.shape)

In [0]:
#lookup structures
# mapping item index
product_id_to_idx = pd.Series(
    pdf_items.index,
    index=pdf_items["product_id"]
).drop_duplicates()

idx_to_product_id = pd.Series(
    pdf_items["product_id"].values,
    index=pdf_items.index
)

In [0]:
#convert train/test to pandas
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()

print("train rows:", train_pd.shape)
print("test rows:", test_pd.shape)

In [0]:
#build user interaction dictionaries
train_user_items = (
    train_pd.groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

test_user_items = (
    test_pd.groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

In [0]:
#helper function: recommend items for a product
def similar_items(product_id, top_k=20):
    
    if product_id not in product_id_to_idx:
        return []
    
    idx = product_id_to_idx[product_id]
    
    scores = list(enumerate(cosine_sim[idx]))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    scores = scores[1: top_k+1]   # exclude itself
    
    return [
        idx_to_product_id[i]
        for i, s in scores
    ]

In [0]:
#build recommendations for each user. We combine recommendations from all items in the user's train set.
def recommend_for_user(user_id, k=10):
    
    if user_id not in train_user_items:
        return []
    
    seen_items = train_user_items[user_id]
    
    candidate_scores = {}
    
    for item in seen_items:
        
        recs = similar_items(item, top_k=50)
        
        for rank, rec in enumerate(recs):
            
            if rec in seen_items:
                continue
            
            candidate_scores[rec] = candidate_scores.get(rec, 0) + 1/(rank+1)
    
    
    ranked_items = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    return [i for i, s in ranked_items[:k]]

In [0]:
# Evaluation metric helpers
def precision_at_k(actual, predicted, k):
    predicted = predicted[:k]
    if len(predicted) == 0:
        return 0
    hits = len(set(predicted).intersection(actual))
    return hits / k

def recall_at_k(actual, predicted, k):
    predicted = predicted[:k]
    if len(actual) == 0:
        return 0
    hits = len(set(predicted).intersection(actual))
    return hits / len(actual)

def average_precision_at_k(actual, predicted, k):
    score = 0
    hits = 0
    for i, p in enumerate(predicted[:k]):
        if p in actual:
            hits += 1
            score += hits / (i + 1)
    if hits == 0:
        return 0
    return score / min(len(actual), k)

def dcg_at_k(actual, predicted, k):
    dcg = 0.0
    actual_set = set(actual)
    for i, p in enumerate(predicted[:k]):
        rel = 1 if p in actual_set else 0
        dcg += rel / np.log2(i + 2)
    return dcg

def ndcg_at_k(actual, predicted, k):
    ideal_len = min(len(actual), k)
    if ideal_len == 0:
        return 0
    ideal_dcg = sum(1 / np.log2(i + 2) for i in range(ideal_len))
    if ideal_dcg == 0:
        return 0
    return dcg_at_k(actual, predicted, k) / ideal_dcg

In [0]:
#Recall@K
def recall_at_k(actual, predicted, k):
    
    predicted = predicted[:k]
    
    if len(actual) == 0:
        return 0
    
    hits = len(
        set(predicted).intersection(actual)
    )
    
    return hits / len(actual)

In [0]:
#Average Precision@K
def average_precision_at_k(actual, predicted, k):
    
    score = 0
    hits = 0
    
    for i, p in enumerate(predicted[:k]):
        
        if p in actual:
            
            hits += 1
            
            score += hits / (i+1)
    
    
    if hits == 0:
        return 0
    
    return score / min(len(actual), k)

In [0]:
# Random baseline recommender
import random

all_product_ids = set(pdf_items["product_id"].tolist())

def recommend_random_for_user(user_id, k=10):
    if user_id not in train_user_items:
        return []
    seen_items = train_user_items[user_id]
    candidate_items = list(all_product_ids - seen_items)
    if len(candidate_items) == 0:
        return []
    if len(candidate_items) < k:
        return candidate_items
    return random.sample(candidate_items, k)

In [0]:
# Lift over random compares model Precision@K against random recommendation Precision@K.
K = 10

model_precisions = []
random_precisions = []

for user_id in test_user_items:
    actual_items = test_user_items[user_id]

    pred_model = recommend_for_user(user_id, k=K)
    pred_random = recommend_random_for_user(user_id, k=K)

    model_precisions.append(precision_at_k(actual_items, pred_model, K))
    random_precisions.append(precision_at_k(actual_items, pred_random, K))

model_precision_k = np.mean(model_precisions)
random_precision_k = np.mean(random_precisions)

lift_over_random = (
    model_precision_k / random_precision_k
    if random_precision_k > 0 else np.nan
)

print(f"Model Precision@{K}: {model_precision_k:.4f}")
print(f"Random Precision@{K}: {random_precision_k:.4f}")
print(f"Lift over random: {lift_over_random:.4f}")

In [0]:
K = 10

model_precisions = []
random_precisions = []

for user_id in test_user_items:
    actual_items = test_user_items[user_id]
    
    pred_model = recommend_for_user(user_id, k=K)
    pred_random = recommend_random_for_user(user_id, k=K)
    
    model_precisions.append(precision_at_k(actual_items, pred_model, K))
    random_precisions.append(precision_at_k(actual_items, pred_random, K))

model_precision_k = np.mean(model_precisions)
random_precision_k = np.mean(random_precisions)

lift_over_random = (
    model_precision_k / random_precision_k
    if random_precision_k > 0 else np.nan
)

print(f"Model Precision@{K}: {model_precision_k:.4f}")
print(f"Random Precision@{K}: {random_precision_k:.4f}")
print(f"Lift over random: {lift_over_random:.4f}")

In [0]:
# Inter-user diversity.
# Higher value = users get more different recommendation lists.
from itertools import combinations

def jaccard_similarity(set_a, set_b):
    union = set_a.union(set_b)
    if len(union) == 0:
        return 0
    return len(set_a.intersection(set_b)) / len(union)

In [0]:
K = 10

# Faster version: estimate inter-user diversity on a sample of users
# instead of all pairwise combinations across the full test population.
MAX_DIVERSITY_USERS = 300

sample_user_ids = list(test_user_items.keys())[:MAX_DIVERSITY_USERS]

user_recs = {}

for user_id in sample_user_ids:
    recs = recommend_for_user(user_id, k=K)
    user_recs[user_id] = set(recs)

user_ids = list(user_recs.keys())
pairwise_jaccards = []

for u1, u2 in combinations(user_ids, 2):
    sim = jaccard_similarity(user_recs[u1], user_recs[u2])
    pairwise_jaccards.append(sim)

avg_jaccard = np.mean(pairwise_jaccards) if len(pairwise_jaccards) > 0 else 0
inter_user_diversity = 1 - avg_jaccard

print(f"Average Jaccard similarity across sampled users: {avg_jaccard:.4f}")
print(f"Inter-user diversity (sampled): {inter_user_diversity:.4f}")


In [0]:
#build popularity baseline
train_pd = spark.table("workspace.default.sephora_interactions_train").toPandas()

popular_items = (
    train_pd["product_id"]
    .value_counts()
    .head(50)
    .index
    .tolist()
)

popular_item_set = set(popular_items)

print("Top popular items baseline size:", len(popular_item_set))

In [0]:
#compute serendipity@K
def serendipity_at_k(actual, predicted, popular_baseline, k):
    predicted_k = predicted[:k]
    
    if len(predicted_k) == 0:
        return 0
    
    serendipitous_hits = [
        item for item in predicted_k
        if (item in actual) and (item not in popular_baseline)
    ]
    
    return len(serendipitous_hits) / k

In [0]:
K = 10

serendipities = []

for user_id in test_user_items:
    actual_items = test_user_items[user_id]
    predicted_items = recommend_for_user(user_id, k=K)
    
    s = serendipity_at_k(
        actual=actual_items,
        predicted=predicted_items,
        popular_baseline=popular_item_set,
        k=K
    )
    
    serendipities.append(s)

mean_serendipity = np.mean(serendipities)

print(f"Serendipity@{K}: {mean_serendipity:.4f}")

In [0]:
# Run evaluation
K = 10

precisions = []
recalls = []
aps = []
ndcgs = []

all_recommended_items = set()

# Proxy error metrics for consistency with your other notebooks.
# Caveat:
# - This content filter ranks by similarity; it does not directly predict ratings.
# - So MAE / RMSE below are proxy ranking-to-score errors, not strict rating-prediction errors.
proxy_actuals = []
proxy_preds = []

# FIX: ensure rating is numeric
test_pd["rating"] = pd.to_numeric(test_pd["rating"], errors="coerce")

# Build quick lookup for test ratings
test_rating_lookup = (
    test_pd.groupby(["author_id", "product_id"])["rating"]
    .mean()
    .to_dict()
)

for user_id in test_user_items:
    actual_items = test_user_items[user_id]
    predicted_items = recommend_for_user(user_id, k=K)

    all_recommended_items.update(predicted_items)

    precisions.append(precision_at_k(actual_items, predicted_items, K))
    recalls.append(recall_at_k(actual_items, predicted_items, K))
    aps.append(average_precision_at_k(actual_items, predicted_items, K))
    ndcgs.append(ndcg_at_k(actual_items, predicted_items, K))

    # proxy predicted score: descending rank score for top-K, else 0
    pred_score_map = {
        item: (K - rank) / K
        for rank, item in enumerate(predicted_items[:K])
    }

    for item in actual_items:
        actual_rating = test_rating_lookup.get((user_id, item), 0)
        actual_scaled = actual_rating / 5.0
        pred_scaled = pred_score_map.get(item, 0.0)

        proxy_actuals.append(actual_scaled)
        proxy_preds.append(pred_scaled)

In [0]:
# Compute final metrics
precision_k = np.mean(precisions)
recall_k = np.mean(recalls)
map_k = np.mean(aps)
ndcg_k = np.mean(ndcgs)

coverage = len(all_recommended_items) / len(pdf_items)

mae_proxy = mean_absolute_error(proxy_actuals, proxy_preds) if len(proxy_actuals) > 0 else np.nan
rmse_proxy = np.sqrt(mean_squared_error(proxy_actuals, proxy_preds)) if len(proxy_actuals) > 0 else np.nan

print("RESULTS")
print("----------------------")
print(f"Precision@{K}: {precision_k:.4f}")
print(f"Recall@{K}: {recall_k:.4f}")
print(f"MAP@{K}: {map_k:.4f}")
print(f"NDCG@{K}: {ndcg_k:.4f}")
print(f"Catalogue coverage: {coverage:.4f}")

print(f"Lift over random: {lift_over_random:.4f}")
print(f"Inter-user diversity (sampled): {inter_user_diversity:.4f}")
print(f"Serendipity@{K}: {mean_serendipity:.4f}")

print("\nProxy error metrics (not directly comparable to CF/MF rating-prediction RMSE / MAE)")
print(f"MAE (proxy): {mae_proxy:.4f}")
print(f"RMSE (proxy): {rmse_proxy:.4f}")
